In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("data/crocodile_dataset.csv")

df.info()

In [ ]:

df.drop(columns=['Notes'], axis=1, inplace=True)
df.head(20)

In [ ]:
df["Age Class"].value_counts()

In [ ]:
# OneHotEncode
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

categorical_features=["Common Name", "Scientific Name", "Family", "Genus", "Sex", "Country/Region", "Habitat Type", "Conservation Status", "Observer Name"]
ordinal_categorical_features=["Age Class"]


encoder = OneHotEncoder(sparse_output=False)
one_hot_encoded = encoder.fit_transform(df[categorical_features])
one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(categorical_features))
df_sklearn_encoded = pd.concat([df.drop(categorical_features, axis=1), one_hot_df], axis=1)

le = LabelEncoder()
df_sklearn_encoded["Age Class"] = le.fit_transform(df_sklearn_encoded[ordinal_categorical_features])

df_sklearn_encoded['Date of Observation'] = pd.to_datetime(df_sklearn_encoded['Date of Observation'], format='%d-%m-%Y')
df_sklearn_encoded['DaysSinceEpoch'] = (df_sklearn_encoded['Date of Observation'] - pd.Timestamp('1970-01-01')).dt.days
df_sklearn_encoded.drop('Date of Observation', inplace=True, axis=1)

df_sklearn_encoded.head()


# print(f"One-Hot Encoded Data using Scikit-Learn:\n{df_sklearn_encoded}\n")

# print(df_sklearn_encoded.columns)

In [ ]:

from sklearn.model_selection import train_test_split

X = df_sklearn_encoded.drop(["Age Class"], axis=1)
y = df_sklearn_encoded["Age Class"]


X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42, test_size=0.2)

print(type(df_sklearn_encoded))

print(type(X))
print(type(X_train))

print(type(y))
print(type(y_train))




In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


dtc = DecisionTreeClassifier(criterion='gini')

dtc.fit(X_train, y_train)

print(accuracy_score(y_test, dtc.predict(X_test)))




In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


rand = RandomForestClassifier(random_state=42, class_weight='balanced')

rand.fit(X_train, y_train)
y_pred = dtc.predict(X_test)


print(accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.metrics import classification_report

report_dict = classification_report(y_test, y_pred, output_dict=True)


report_df = pd.DataFrame(report_dict).transpose()
report_df = report_df.drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix


plt.figure(figsize=(10, 5))
sns.heatmap(report_df[['precision', 'recall', 'f1-score']], annot=True, cmap='YlGnBu', fmt=".2f")
plt.title("Classification Report Heatmap")
plt.ylabel("Classes")
plt.xlabel("Metrics")
plt.tight_layout()
plt.show()
